# Mae Na Rua + WMB Phayao — Daily Pipeline (Colab)

รันเรียงจากบนลงล่างทุกครั้งที่เปิด session ใหม่ (ห้ามกด "Runtime > Run all" ข้ามจากอันเก่า —
เซลล์ทดสอบ Phase 0-5 (ERA5T decode test, GEE auth test, MEI/CHIRPS เดี่ยว, SAR classification เต็ม)
ถูกลบออกจาก notebook นี้แล้ว เพราะเป็นการทดสอบระหว่างย้ายระบบที่ผ่านแล้ว ไม่ต้องรันซ้ำ — โค้ดเดิมยังอยู่ใน
`colab_migration/COLAB_MIGRATION_PLAN.md` ถ้าต้องย้อนดู

รายละเอียด/troubleshooting เต็มๆ ดูที่ `colab_migration/DAILY_RUNBOOK.md`

In [ ]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# Cell 1 — mount Drive + path constants

from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE = "/content/drive/MyDrive/Colab Notebooks/Mae_Na_Rua"
PROJECT_WEB = f"{DRIVE_BASE}/maenaruea-water-web"
PROJECT_WMB = f"{DRIVE_BASE}/WMB_Phayao"
PIPELINE_DIR = f"{PROJECT_WEB}/01_data/scripts and code/pipeline"          # อ่านอย่างเดียว ห้ามเขียน
COLAB_MIGRATION_DIR = f"{PROJECT_WEB}/01_data/scripts and code/colab_migration"
WMB_COLAB_MIGRATION_DIR = f"{PROJECT_WMB}/colab_migration"

for p in [PROJECT_WEB, PIPELINE_DIR, COLAB_MIGRATION_DIR, PROJECT_WMB, WMB_COLAB_MIGRATION_DIR]:
    print(p, "->", "OK" if os.path.exists(p) else "!! ไม่พบ ตรวจสอบ path/การ mount")

In [ ]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# ติดตั้ง dependency ของ ERA5T

!pip install -q cdsapi cfgrib eccodes ecmwflibs xarray
import eccodes
print("eccodes version:", eccodes.codes_get_api_version())

In [ ]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# โหลด CDS credential จาก Colab Secret (CDSAPI_URL / CDSAPI_KEY) เป็น .cdsapirc

from google.colab import userdata

cdsapi_url = userdata.get('CDSAPI_URL')
cdsapi_key = userdata.get('CDSAPI_KEY')

with open('/root/.cdsapirc', 'w') as f:
    f.write(f"url: {cdsapi_url}\nkey: {cdsapi_key}\n")
print("เขียน .cdsapirc แล้ว")

In [ ]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# เพิ่ม path สำหรับ import โมดูลที่พอร์ตแล้ว/ต้นฉบับ

import sys
sys.path.insert(0, PIPELINE_DIR)
sys.path.insert(0, COLAB_MIGRATION_DIR)

In [ ]:
import importlib
import data_pipeline_colab as dp
importlib.reload(dp)

result = dp.backfill_historical_weeks(
    start_year=2026, start_week=15,
    end_year=2026, end_week=26,
)
print("backfilled:", len(result["backfilled"]))
print("already_complete_skipped:", len(result["already_complete_skipped"]))
print("too_recent_skipped:", len(result["too_recent_skipped"]))
print("still_incomplete:", result["still_incomplete"])
print("errors:", result["errors"])

In [ ]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# ตั้ง GEE Service Account จาก Colab Secret

!pip install -q earthengine-api

from google.colab import userdata
import os

gee_sa_key_json = userdata.get('GEE_SA_KEY_JSON')
gee_sa_email = userdata.get('GEE_SA_EMAIL')

gee_key_path = "/content/gee_sa_key.json"
with open(gee_key_path, "w") as f:
    f.write(gee_sa_key_json)

os.environ["GEE_SERVICE_ACCOUNT_EMAIL"] = gee_sa_email
os.environ["GEE_SERVICE_ACCOUNT_KEY"] = gee_key_path

print("ตั้ง env var แล้วสำหรับ:", gee_sa_email)

In [ ]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# dependency โมเดลทำนาย (Water Demand / Reservoir Inflow)
# 2026-07-22 เพิ่ม rasterio: chirps_feature.py มี tier ใหม่ "CHIRPS Prelim FTP" ที่อ่านไฟล์
# .tif ตรงจาก CHC FTP ด้วย rasterio (ดึงข้อมูลเร็วกว่า Prelim เดิม ~3 เท่า, lag ~7 วัน) --
# ถ้าไม่ติดตั้ง จะไม่ error แต่จะ silent fallback ไป tier Prelim เดิม (community, ช้ากว่า)

!pip install -q catboost==1.2.10 lightgbm==4.6.0 openpyxl==3.1.5 rasterio

In [ ]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# dependency ของ daily_update_colab.py (WMB_Phayao) — requests: โหลด gdrive_log, plotly: rebuild Reservoirs_inflow.html

!pip install -q requests openpyxl plotly

## รันประจำวันจากนี้ลงไป (ลำดับสำคัญ — WMB_Phayao ก่อน Mae Na Rua ก่อน push)

In [ ]:
#✅ ทุกวัน
# รัน WMB_Phayao daily_update_colab.py (พยากรณ์น้ำท่วม 7 วัน + inflow อ่าง 5 อ่าง)
# ห้ามใส่ --offline (นั่นคือโหมดทดสอบ ไม่ดึงข้อมูลจริง)

import subprocess, os

r = subprocess.run(
    ["python3", f"{WMB_COLAB_MIGRATION_DIR}/daily_update_colab.py"],
    env={**os.environ, "WMB_ROOT": PROJECT_WMB},
    capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr[-2000:])

In [ ]:
#✅ ทุกวัน
# รัน Mae Na Rua หลัก — climate features (MEI/CHIRPS/ERA5T) -> อ่าน SAR จากแคช -> ทำนาย -> เขียน latest.json

import importlib
import data_pipeline_colab as dp
importlib.reload(dp)

result = dp.run_pipeline()
print("status:", result.status)
print("step_status:", result.step_status)
print("errors:", result.errors)

In [ ]:
#✅ ทุกวัน
# push ไฟล์ข้อมูล 4 ตัวขึ้น GitHub (latest.json, flood_latest.json, reservoir_inflow.json, flood_depth_forecast.png)

from google.colab import userdata
import subprocess, os, shutil
from pathlib import Path
from datetime import datetime

GITHUB_PAT = userdata.get('GITHUB_PAT')
GITHUB_REPO_URL = f"https://{GITHUB_PAT}@github.com/mpdox30/maenarua-water-web.git"

PUSH_CLONE_DIR = "/content/repo_push"     # ephemeral — clone สดใหม่ทุกรอบ ไม่ผูกกับ Drive .git ใดๆ
GIT_USER_NAME = "Mae Na Rua Pipeline (Colab)"
GIT_USER_EMAIL = "mp.dox69@gmail.com"

# เฉพาะไฟล์ที่ Colab ดูแล auto-push — ไฟล์อื่น (HTML/โค้ด) ผู้ใช้ push เองจาก Windows
FILES_TO_PUSH = [
    "03_website/assets/data/latest.json",
    "03_website/assets/data/flood_latest.json",
    "03_website/assets/data/reservoir_inflow.json",
    "03_website/assets/data/flood_depth_forecast.png",  # 2026-07-26 เพิ่ม -- พยากรณ์น้ำท่วม 7 วัน
]

def push_daily_data():
    if os.path.exists(PUSH_CLONE_DIR):
        shutil.rmtree(PUSH_CLONE_DIR)
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, PUSH_CLONE_DIR],
                   check=True, capture_output=True, text=True)
    subprocess.run(["git", "-C", PUSH_CLONE_DIR, "config", "user.name", GIT_USER_NAME], check=True)
    subprocess.run(["git", "-C", PUSH_CLONE_DIR, "config", "user.email", GIT_USER_EMAIL], check=True)

    changed = []
    for rel in FILES_TO_PUSH:
        src = Path(PROJECT_WEB) / rel
        dst = Path(PUSH_CLONE_DIR) / rel
        if not src.exists():
            print(f"ข้าม (ไม่พบไฟล์ต้นทางบน Drive): {rel}")
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        changed.append(rel)

    if not changed:
        print("ไม่มีไฟล์ให้ push เลย — เช็ค PROJECT_WEB/path ก่อน")
        return

    subprocess.run(["git", "-C", PUSH_CLONE_DIR, "add"] + changed, check=True)
    diff = subprocess.run(["git", "-C", PUSH_CLONE_DIR, "diff", "--cached", "--stat"],
                         capture_output=True, text=True)
    if not diff.stdout.strip():
        print("ข้อมูลไม่เปลี่ยนจากรอบก่อน (เทียบกับ HEAD ของ repo) — ไม่ commit/push")
        return

    msg = f"Auto-update: pipeline data {datetime.now().strftime('%Y-%m-%d %H:%M')}"
    subprocess.run(["git", "-C", PUSH_CLONE_DIR, "commit", "-m", msg], check=True)
    result = subprocess.run(["git", "-C", PUSH_CLONE_DIR, "push", "origin", "HEAD:master"],
                            capture_output=True, text=True)
    if result.returncode == 0:
        print("push สำเร็จ:", msg)
    else:
        print("push ไม่สำเร็จ:")
        print(result.stderr[-800:])

push_daily_data()